# Stage 2: gemma extraction
Dedicated model launcher using the same shared extraction implementation and numerical policy as extract_features.ipynb.
Use a separate L4 runtime from the other model. Run cells in order. Do not start a second job for this same model while its current job is running.
The accepted model plan is prefilled from the user-reported preparation output and verified on launch. For a resumed job, supply its latest explicit RESUME_INDEXES before execution; never use another model's checkpoint.


## Colab setup
Run this setup first on a fresh runtime. It mounts Drive, obtains the integrated phase1 checkout, adds its import path and installs the appropriate dependency profile. Existing checkouts are never reset or updated automatically.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/content/Research')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO_URL = 'https://github.com/Drazr/Research.git'
BRANCH = 'phase1'
STORAGE_SCOPE = 'persistent-drive'

try:
    from google.colab import drive
except ImportError:
    if STORAGE_SCOPE != 'local-development':
        raise RuntimeError('Outside Colab, set explicit paths and local-development scope.')
else:
    drive.mount('/content/drive')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError('Mount the intended Drive containing the accepted phase1 artifacts.')
if not (REPO / '.git').is_dir():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError('Repository destination is nonempty; choose a clean checkout.')
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', '-c', 'core.autocrlf=false', 'clone', '--branch', BRANCH,
                    REPO_URL, str(REPO)], check=True)

def checkout_value(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()

if checkout_value('remote', 'get-url', 'origin').removesuffix('.git').rstrip('/') != REPO_URL.removesuffix('.git'):
    raise RuntimeError('Expected Drazr/Research checkout.')
if checkout_value('branch', '--show-current') != BRANCH or checkout_value('status', '--porcelain'):
    raise RuntimeError('Use a clean phase1 checkout; do not switch or pull during a running job.')
for filename in ('interfaces.py', 'extract_features.py', 'prepare_e03.py'):
    if not (REPO / 'src/stage2' / filename).is_file():
        raise RuntimeError('This checkout predates Stage 2 integration; update it before setup.')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ['PHASE1_ROOT'] = str(PHASE1_ROOT)
profile = REPO / 'requirements-l4.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(profile)], check=True)
print('Frozen implementation commit:', checkout_value('rev-parse', 'HEAD'))
print('Persistent root:', PHASE1_ROOT)


In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

MODEL = 'gemma'  # Dedicated gemma launcher; backend logic remains shared.
MODEL_PLAN = str(PHASE1_ROOT / 'cache/shared/stage2-plans/20261115T115910Z-57256c0a6c824feab6d2c19f479d5d0a/_COMPLETE.json')
RESUME_INDEXES = []  # Explicit printed progress _COMPLETE.json paths; no newest-file lookup.
# All model/config/data/readiness settings are resolved from MODEL_PLAN by the worker.
# Execution always repeats preflight; rerunning this cell cannot retain an old pass.

## Mount the intended Drive and freeze the checkout
Authenticate to the intended Drive account. Existing checkouts are checked and never
pulled automatically. After an intentional code update restart setup; never update
code mid-run. Pin the printed code commit when reproducing a run.

In [ ]:
try:
    from google.colab import drive
except ImportError:
    if STORAGE_SCOPE != 'local-development':
        raise RuntimeError('Outside Colab, supply an explicit development root and scope.')
else:
    drive.mount('/content/drive')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Intended Drive mount is unavailable.')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError('PHASE1_ROOT must already exist on the intended mounted storage.')
if not (REPO/'.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError('Choose an empty repository destination.')
    subprocess.run(['git', '-c', 'core.autocrlf=false', 'clone', '--branch', BRANCH,
                    REPO_URL, str(REPO)], check=True)
def git_output(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()
if git_output('remote', 'get-url', 'origin').removesuffix('.git') != REPO_URL.removesuffix('.git'):
    raise RuntimeError('Expected Drazr/Research checkout.')
if git_output('status', '--porcelain'):
    raise RuntimeError('Use a clean frozen checkout.')
subprocess.run(['git', '-C', str(REPO), 'merge-base', '--is-ancestor',
                '986f812ac6e3811a035391738e295bf55043369a', 'HEAD'], check=True)
if not (REPO/'src/stage2/extract_features.py').is_file():
    raise RuntimeError('Checkout does not contain the extraction worker.')
CODE_COMMIT = git_output('rev-parse', 'HEAD')
os.environ['PHASE1_ROOT'] = str(PHASE1_ROOT)
print('Code commit:', CODE_COMMIT)
print('Storage root:', PHASE1_ROOT)

## Install the selected runtime profile
Use the pinned requirements in a fresh Python 3.11/3.12 L4 runtime. Restart after
installation if Colab requests it, then rerun setup. Do not upgrade packages or
change precision, attention, image policy or token limits to work around a failure.

In [ ]:
# Dependencies were installed by the common bootstrap cell.

In [ ]:
def redact(text):
    secret = os.environ.get('HF_TOKEN')
    if secret:
        text = text.replace(secret, '[REDACTED]')
    return re.sub(r'hf_[A-Za-z0-9]+', '[REDACTED]', text)

def run_worker(preflight=False):
    if MODEL != 'gemma':
        raise RuntimeError('This launcher is for gemma; use the other model notebook instead.')
    if 'PASTE_' in str(MODEL_PLAN) or not str(MODEL_PLAN).endswith('_COMPLETE.json'):
        raise RuntimeError('Set the printed model plan completion path first.')
    if git_output('rev-parse', 'HEAD') != CODE_COMMIT or git_output('status', '--porcelain'):
        raise RuntimeError('Checkout changed since setup; use a clean fixed commit.')
    command = [sys.executable, '-m', 'src.stage2.extract_features', '--model', MODEL,
               '--phase1-root', str(PHASE1_ROOT), '--plan-manifest', str(MODEL_PLAN),
               '--storage-scope', STORAGE_SCOPE]
    for index in RESUME_INDEXES:
        command += ['--resume-index', str(index)]
    if preflight:
        command += ['--preflight']
    report = None
    with subprocess.Popen(command, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(redact(line), end='', flush=True)
            try:
                candidate = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(candidate, dict) and 'status' in candidate:
                report = candidate
        code = process.wait()
    if code != 0 or report is None or report.get('status') != 'passed':
        raise RuntimeError('Extraction/preflight stopped. Preserve artifacts and the printed resume index; resolve the diagnostic before continuing.')
    return report

# Fully verify the plan/data/config/readiness and explicit resume references before weight loading.
PREFLIGHT_REPORT = run_worker(preflight=True)

## Gemma authentication in this runtime
Enable notebook access to the HF_TOKEN Colab secret for a read-capable account with
Gemma access. An existing HF_TOKEN environment value takes precedence; explicitly
replace a stale value through a hidden prompt or the secret store. Tokens are never
printed, embedded in plans or saved in notebook outputs. Qwen needs no token.

In [ ]:
if MODEL == 'gemma':
    from getpass import getpass
    from huggingface_hub import login
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass('Hugging Face read token for your Gemma-authorized account: ')
    if not token:
        raise RuntimeError('Gemma authentication is required.')
    os.environ['HF_TOKEN'] = token
    try:
        login(token=token, add_to_git_credential=False)
    except Exception:
        raise RuntimeError('Hugging Face authentication failed; check runtime secret and model access.') from None
    finally:
        del token

## Extract and publish
The worker rechecks settings, loads one model once, checks actual runtime identity,
tests storage in another process, compares all nine first-FIT views to the public
path, and processes independent 32-image storage chunks with tensor batch one.
Every combined context gets a visual audit; every condition chunk gets first/last
repeats. No inputs are truncated, substituted or dropped. CONFIRM stays closed.

Copy printed resume-index references after interruptions. Set RESUME_INDEXES and
rerun setup/preflight/execution; only checksum-verified compatible chunks are skipped.
Corrupt attempts remain untouched. Numerical conflicts block release. Complete FIT
and DEV-S releases can be handed off while other partitions continue. A passed
worker run covers one model and does not establish whole-stage completion or G0.

In [ ]:
# Fresh process and fresh preflight every launch; no stale in-memory success flag.
run_worker(preflight=True)
EXTRACTION_REPORT = run_worker()